###Loading Prepared Data

In [ ]:
import os
import pandas as pd

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
notebook_path = '/content/drive/My Drive/Python Projects/Walmart Sales Forecast'
save_path = os.path.join(notebook_path, 'data')

In [ ]:
df_imp = os.path.join(save_path, "df_imp.csv")
final_df = os.path.join(save_path, "final_df.csv")
test_file = os.path.join(save_path, "test_merged.csv")

In [ ]:
df1 = pd.read_csv(df_imp)
df2 = pd.read_csv(final_df)
test = pd.read_csv(test_file)

Check

In [ ]:
df1.shape

(420285, 15)

In [ ]:
df1.head()

,Store,Dept,Date,Weekly_Sales,Type,DayOfWeek,SalesPerSize,Store_Mean_Sales_4w,Store_Mean_Sales_12w,Store_Std_Sales_4w,Sales_rollstd_4,StoreDept,Holiday_Dept,Holiday_Month,LogWeeklySales
0,1,1,2010-02-05,24924.50,0,4,0.164719,24924.5000,24924.5000,NaN,NaN,1_1,1_0,2_0,10.123647
1,1,2,2010-02-05,50605.27,0,4,0.334437,37764.8850,37764.8850,18159.046613,NaN,1_2,2_0,2_0,10.831831
2,1,3,2010-02-05,13740.12,0,4,0.090805,29756.6300,29756.6300,18901.638325,NaN,1_3,3_0,2_0,9.528148
3,1,4,2010-02-05,39954.04,0,4,0.264045,32305.9825,32305.9825,16253.555927,NaN,1_4,4_0,2_0,10.595510
4,1,5,2010-02-05,32229.38,0,4,0.212995,34132.2025,32290.6620,15542.559923,NaN,1_5,5_0,2_0,10.380665


In [ ]:
df2.shape

(420285, 5)

In [ ]:
df2.head()

,Type,SalesPerSize,Store_Mean_Sales_4w,Store_Std_Sales_4w,Weekly_Sales
0,0,0.164719,24924.5000,NaN,24924.50
1,0,0.334437,37764.8850,18159.046613,50605.27
2,0,0.090805,29756.6300,18901.638325,13740.12
3,0,0.264045,32305.9825,16253.555927,39954.04
4,0,0.212995,34132.2025,15542.559923,32229.38


In [ ]:
test.shape

(115064, 16)

In [ ]:
test.head()

,Store,Dept,Date,IsHoliday_x,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,IsHoliday_y,Type,Size
0,1,1,2012-11-02,0,55.32,3.386,0.0,0.0,0.0,0.0,0.0,223.462779,6.573,0,0,151315
1,1,1,2012-11-09,0,61.24,3.314,0.0,0.0,0.0,0.0,0.0,223.481307,6.573,0,0,151315
2,1,1,2012-11-16,0,52.92,3.252,0.0,0.0,0.0,0.0,0.0,223.512911,6.573,0,0,151315
3,1,1,2012-11-23,1,56.23,3.211,0.0,0.0,0.0,0.0,0.0,223.561947,6.573,1,0,151315
4,1,1,2012-11-30,0,52.34,3.207,0.0,0.0,0.0,0.0,0.0,223.610984,6.573,0,0,151315


###Import and Installations

In [ ]:
!pip install -U scikit-learn xgboost statsmodels -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 46.8 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import itertools
import cudf

from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from statsmodels.tsa.forecasting.stl import STLForecast
from statsmodels.tsa.holtwinters import ExponentialSmoothing

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

/usr/local/lib/python3.12/dist-packages/cudf/utils/gpu_utils.py:75: UserWarning: Failed to dlopen libcuda.so.1
  warnings.warn(str(e))


###Train-Test Split

In [ ]:
def get_train_test(df, target="Weekly_Sales", test_size=0.2, date_col="Date"):
    df = df.copy()
    df = df.sort_values(by=date_col) if date_col in df.columns else df
    split_idx = int(len(df) * (1 - test_size))
    train, test = df.iloc[:split_idx], df.iloc[split_idx:]
    return train, test

###Evaluate Model Performance

In [ ]:
def evaluate_model(y_true, y_pred, model_name, weights=None):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)

    if weights is None:
        weights = np.ones_like(y_true)

    wmae = np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights)

    print(f"{model_name} → RMSE: {rmse:.2f}, R²: {r2:.4f}, WMAE: {wmae:.2f}")
    return rmse, r2, wmae

####Features in df1 & df2

In [ ]:
features_df1 = [col for col in df1.columns if col not in ['Weekly_Sales', 'Date']]

In [ ]:
features_df2 = [col for col in df2.columns if col not in ['Weekly_Sales', 'Date']]

###Random Forest Regressor

In [ ]:
def run_random_forest(df, features, target="Weekly_Sales", date_col="Date"):
    train, test = get_train_test(df, target, date_col=date_col)
    X_train, y_train = train[features], train[target]
    X_test, y_test = test[features], test[target]

    rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    preds = rf.predict(X_test)

    # Holiday weights
    if "IsHoliday_x" in test.columns:
        weights = np.where(test["IsHoliday_x"].astype(bool).values, 5, 1)
    else:
        weights = np.ones_like(y_test)

    rmse, r2, wmae = evaluate_model(y_test, preds, "Random Forest", weights)
    return rf, train, test, preds, rmse, r2, wmae

In [ ]:
# Run and evaluate on df1
print("Evaluating Random Forest on df1:")
rf_df1, train_df1, test_df1, preds_rf_df1, rf_rmse_df1, rf_r2_df1, rf_wmae_df1 = run_random_forest(df1, features_df1)

Evaluating Random Forest on df1:
Random Forest → RMSE: 3.88, R²: 1.0000, WMAE: 0.42


In [ ]:
# Run and evaluate on df2
print("Evaluating Random Forest on df2:")
rf_df2, train_df2, test_df2, preds_rf_df2, rf_rmse_df2, rf_r2_df2, rf_wmae_df2 = run_random_forest(df2, features_df2)

Evaluating Random Forest on df2:
Random Forest → RMSE: 4319.09, R²: 0.9463, WMAE: 1555.39


###XGBoost

In [ ]:
def run_xgboost(df, features, target="Weekly_Sales", date_col="Date"):
    train, test = get_train_test(df, target, date_col=date_col)

    test_original = test.copy()

    # Convert pandas → cudf
    X_train, y_train = cudf.DataFrame(train[features].copy()), cudf.Series(train[target].values)
    X_test, y_test = cudf.DataFrame(test[features].copy()), cudf.Series(test[target].values)

    # Handle datetime features
    for col in X_train.select_dtypes(include=["datetime64[ns]"]).columns:
        X_train[col+"_year"] = X_train[col].dt.year
        X_train[col+"_month"] = X_train[col].dt.month
        X_test[col+"_year"] = X_test[col].dt.year
        X_test[col+"_month"] = X_test[col].dt.month
        X_train = X_train.drop(columns=[col])
        X_test = X_test.drop(columns=[col])

    # Handle categorical features
    for col in X_train.select_dtypes(include=["object"]).columns:
        X_train[col] = X_train[col].astype("category").cat.codes
        X_test[col] = X_test[col].astype("category").cat.codes

    # Fill missing values (required for cudf → cupy conversion)
    X_train = X_train.fillna(-999)
    X_test = X_test.fillna(-999)
    y_train = y_train.fillna(y_train.mean())
    y_test = y_test.fillna(y_test.mean())

    # cudf → cupy
    X_train_cupy = X_train.to_cupy()
    y_train_cupy = y_train.to_cupy()
    X_test_cupy = X_test.to_cupy()
    y_test_cupy = y_test.to_cupy()

    # XGBoost GPU
    xgb = XGBRegressor(
        n_estimators=300,
        learning_rate=0.1,
        max_depth=6,
        random_state=42,
        n_jobs=-1,
        tree_method="hist",
        device="cuda",
        enable_categorical=True
    )

    xgb.fit(X_train_cupy, y_train_cupy)
    preds = xgb.predict(X_test_cupy)

    if "IsHoliday_x" in test_original.columns:
        weights = np.where(test_original["IsHoliday_x"].values, 5, 1)
    else:
        weights = np.ones_like(y_test_cupy.get())

    rmse, r2, wmae = evaluate_model(y_test_cupy.get(), preds, "XGBoost", weights)
    return xgb, train, test, preds, rmse, r2, wmae

In [ ]:
# Run and evaluate on df1
print("Evaluating XGBoost on df1:")
xgb_df1, train_df1, test_df1, preds_xgb_df1, xgb_rmse_df1, xgb_r2_df1, xgb_wmae_df1 = run_xgboost(df1, features_df1)

Evaluating XGBoost on df1:
XGBoost → RMSE: 420.97, R²: 0.9996, WMAE: 113.26


In [ ]:
# Run and evaluate on df2
print("Evaluating XGBoost on df2:")
xgb_df2, train_df2, test_df2, preds_xgb_df2, xgb_rmse_df2, xgb_r2_df2, xgb_wmae_df2 = run_xgboost(df2, features_df2)

Evaluating XGBoost on df2:
XGBoost → RMSE: 4149.39, R²: 0.9504, WMAE: 1575.30


###STLForecast

In [ ]:
def run_stl_forecast(df, target="Weekly_Sales", date_col="Date"):
    # Aggregate weekly sales
    df_agg = df.groupby(date_col)[target].sum().reset_index()

    # Add holiday flag (aggregate by week: any holiday = holiday)
    if "IsHoliday_x" in df.columns:
        holiday_weekly = (
            df.groupby(date_col)["IsHoliday_x"]
              .max()  # if any holiday in that week → True
              .reset_index()
        )
        df_agg = df_agg.merge(holiday_weekly, on=date_col, how="left")
    else:
        df_agg["IsHoliday_x"] = False

    # Train/test split
    train, test = get_train_test(df_agg, target, date_col=date_col)

    # Ensure datetime index
    train.index = pd.to_datetime(train[date_col])
    test.index = pd.to_datetime(test[date_col])

    # Resample keeping both sales + holiday
    train = train.resample("W").agg({target: "sum", "IsHoliday_x": "max"})
    test = test.resample("W").agg({target: "sum", "IsHoliday_x": "max"})

    # Fill missing values for sales
    train[target] = train[target].interpolate("linear").ffill().bfill()
    test[target] = test[target].interpolate("linear").ffill().bfill()

    # STL + ARIMA
    stlf = STLForecast(
        endog=train[target],
        model=ARIMA,
        model_kwargs={"order": (2, 1, 2)},
        period=52
    )
    stlf_fit = stlf.fit()

    preds = stlf_fit.forecast(len(test))

    # Holiday weights
    weights = np.where(test["IsHoliday_x"].astype(bool).values, 5, 1)

    rmse, r2, wmae = evaluate_model(test[target], preds, "STLForecast", weights)

    return stlf_fit, train, test, preds, rmse, r2, wmae

In [ ]:
# Run and evaluate on df1
print("Evaluating STLForecast on df1:")
stl_df1, train_df1, test_df1, preds_stl_df1, rmse_stl_df1, r2_stl_df1, wmae_stl_df1 = run_stl_forecast(df1)

Evaluating STLForecast on df1:
STLForecast → RMSE: 1886581.30, R²: -0.1882, WMAE: 1443006.25


=> STLForecast failed on df2 because it requires a datetime index.

###Exponential Smoothing

In [ ]:
def run_expsmoothing(df, target="Weekly_Sales", date_col="Date"):
    train, test = get_train_test(df, target, date_col=date_col)
    y_train, y_test = train[target].values, test[target].values

    hw = ExponentialSmoothing(
        y_train,
        trend="add",
        seasonal="add",
        seasonal_periods=12
    )
    hw_fit = hw.fit()
    preds = hw_fit.forecast(len(y_test))

    # Holiday weights
    if "IsHoliday_x" in test.columns:
        weights = np.where(test["IsHoliday_x"].astype(bool).values, 5, 1)
    else:
        weights = np.ones_like(y_test)

    rmse, r2, wmae = evaluate_model(y_test, preds, "Exponential Smoothing", weights)

    return hw_fit, train, test, preds, rmse, r2, wmae

In [ ]:
# Run and evaluate on df1
print("Evaluating Exponential Smoothing on df1:")
hw_df1, train_df1, test_df1, preds_hw_df1, hw_rmse_df1, hw_r2_df1, hw_wmae_df1 = run_expsmoothing(df1)

Evaluating Exponential Smoothing on df1:


/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/holtwinters/model.py:85: RuntimeWarning: overflow encountered in matmul
  return err.T @ err


Exponential Smoothing → RMSE: 22340.23, R²: -0.0315, WMAE: 13979.04


In [ ]:
# Run and evaluate on df2
print("Evaluating Exponential Smoothing on df2:")
hw_df2, train_df2, test_df2, preds_hw_df2, hw_rmse_df2, hw_r2_df2, hw_wmae_df2 = run_expsmoothing(df2)

Evaluating Exponential Smoothing on df2:


/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/holtwinters/model.py:85: RuntimeWarning: overflow encountered in matmul
  return err.T @ err


Exponential Smoothing → RMSE: 270399.12, R²: -209.6525, WMAE: 237559.44


##Model Comparison

In [ ]:
# Results for df1
results_df1 = [
    ["Random Forest", rf_rmse_df1, rf_r2_df1, rf_wmae_df1],
    ["XGBoost", xgb_rmse_df1, xgb_r2_df1, xgb_wmae_df1],
    ["STLForecast", rmse_stl, r2_stl, wmae_stl],
    ["Exponential Smoothing", hw_rmse_df1, hw_r2_df1, hw_wmae_df1]
]

eval_table_df1 = pd.DataFrame(
    results_df1,
    columns=["Model", "RMSE", "R2", "WMAE"]
)

In [ ]:
eval_table_df1

,Model,RMSE,R2,WMAE
0,Random Forest,3.880000e+00,1.0000,0.42
1,XGBoost,4.209700e+02,0.9996,113.26
2,STLForecast,1.886581e+06,-0.1882,1443006.25
3,Exponential Smoothing,-1.882000e-01,-0.0315,13979.04


In [ ]:
# Results for df2
results_df2 = [
    ["Random Forest", rf_rmse_df2, rf_r2_df2, rf_wmae_df2],
    ["XGBoost", xgb_rmse_df2, xgb_r2_df2, xgb_wmae_df2],
    ["STLForecast", None, None, None],
    ["Exponential Smoothing", hw_rmse_df2, hw_r2_df2, hw_wmae_df2]
]

eval_table_df2 = pd.DataFrame(
    results_df2,
    columns=["Model", "RMSE", "R2", "WMAE"]
)

In [ ]:
eval_table_df2

,Model,RMSE,R2,WMAE
0,Random Forest,4319.09,0.9463,1555.39
1,XGBoost,4149.39,0.9504,1575.30
2,STLForecast,NaN,NaN,NaN
3,Exponential Smoothing,270399.12,-209.6525,237559.44


In [ ]:
import plotly.graph_objects as go

# Data
models = ["Random Forest", "XGBoost", "STLForecast", "Exponential Smoothing"]
df1_rmse = [3.88, 420.97, 1886581, -0.1882]
df2_rmse = [4319.09, 4149.39, None, 270399.12]
df1_r2   = [1.0000, 0.9996, -0.1882, -0.0315]
df2_r2   = [0.9463, 0.9504, None, -209.6525]

# RMSE Plot
fig_rmse = go.Figure()
fig_rmse.add_trace(go.Bar(
    x=models,
    y=df1_rmse,
    name="df1 RMSE",
    marker_color="#CDA7F2"
))
fig_rmse.add_trace(go.Bar(
    x=models,
    y=df2_rmse,
    name="df2 RMSE",
    marker_color="#BDE9D8"
))
fig_rmse.update_layout(
    barmode='group',
    title=dict(
        text="Model RMSE Comparison (Log Scale)",
        x=0.5,
        xanchor="center"
    ),
    yaxis_title="RMSE (log)",
    yaxis_type="log"
)

fig_rmse.show()

In [ ]:
# R² Plot
fig_r2 = go.Figure()
fig_r2.add_trace(go.Bar(
    x=models,
    y=df1_r2,
    name="df1 R²",
    marker_color="#CDA7F2"
))
fig_r2.add_trace(go.Bar(
    x=models,
    y=df2_r2,
    name="df2 R²",
    marker_color="#BDE9D8"
))
fig_r2.update_layout(
    barmode='group',
    title=dict(
        text="Model R² Comparison",
        x=0.5,
        xanchor="center"
    ),
    yaxis_title="R²",
    yaxis=dict(range=[-2.5, 1.1])
)

fig_r2.show()

=> Random Forest: near-zero RMSE on df1 = overfitting → unreliable when scaling to new/unseen data

=> XGBoost: generalizes better!
    

*   WMAE: 1575.30
*   RMSE: 4149.39
*   R2: 0.9504




##Saving the Model

**XGBoost** model trained on df2 (dataset refined with Recursive Feature Elimination) ➡ provides more realistic performance and better generalization for forecasting.

In [ ]:
import os
import joblib
import json

In [ ]:
# Path for saving
model_path = '/content/drive/My Drive/Python Projects/Walmart Sales Forecast/models'
os.makedirs(model_path, exist_ok=True)

In [ ]:
final_model = xgb_df2
xgb_model_path = os.path.join(model_path, "xgb_sales_forecast.json")
final_model.save_model(xgb_model_path)

In [ ]:
# Save features
features_path = os.path.join(model_path, "features.json")
with open(features_path, "w") as f:
    json.dump(features_df2, f)

In [ ]:
print("Model and features saved at:", xgb_model_path, "and", features_path)

Model and features saved at: /content/drive/My Drive/Python Projects/Walmart Sales Forecast/models/xgb_sales_forecast.json and /content/drive/My Drive/Python Projects/Walmart Sales Forecast/models/features.json
